In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Notebook setup works")

Notebook setup works


In [3]:
import os
from pathlib import Path

# Ensure working directory is the project root, not the notebooks/ folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

print("Working directory:", Path.cwd())

Working directory: /Users/nateseluga/Downloads/Shot-Value-Machine-Learning


In [4]:
# load dataset
df = pd.read_csv("data/raw/shot_logs.csv")

# basic info
print(df.shape)
df.head()

# check for missing values
df.columns
df.info()
df.isna().sum().sort_values(ascending=False)

(128069, 21)
<class 'pandas.DataFrame'>
RangeIndex: 128069 entries, 0 to 128068
Data columns (total 21 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   GAME_ID                     128069 non-null  int64  
 1   MATCHUP                     128069 non-null  str    
 2   LOCATION                    128069 non-null  str    
 3   W                           128069 non-null  str    
 4   FINAL_MARGIN                128069 non-null  int64  
 5   SHOT_NUMBER                 128069 non-null  int64  
 6   PERIOD                      128069 non-null  int64  
 7   GAME_CLOCK                  128069 non-null  str    
 8   SHOT_CLOCK                  122502 non-null  float64
 9   DRIBBLES                    128069 non-null  int64  
 10  TOUCH_TIME                  128069 non-null  float64
 11  SHOT_DIST                   128069 non-null  float64
 12  PTS_TYPE                    128069 non-null  int64  
 13  SHOT_RESULT 

SHOT_CLOCK                    5567
GAME_ID                          0
SHOT_DIST                        0
player_name                      0
PTS                              0
FGM                              0
CLOSE_DEF_DIST                   0
CLOSEST_DEFENDER_PLAYER_ID       0
CLOSEST_DEFENDER                 0
SHOT_RESULT                      0
PTS_TYPE                         0
TOUCH_TIME                       0
MATCHUP                          0
DRIBBLES                         0
GAME_CLOCK                       0
PERIOD                           0
SHOT_NUMBER                      0
FINAL_MARGIN                     0
W                                0
LOCATION                         0
player_id                        0
dtype: int64

In [5]:
# create working copy and standardize column names
df_clean = df.copy()
df_clean.columns = [c.lower().strip() for c in df_clean.columns]

# define binary target (1 = made, 0 = missed)
df_clean["made"] = df_clean["fgm"]

# convert GAME_CLOCK (MM:SS) to numeric seconds
def clock_to_seconds(x):
    if pd.isna(x):
        return np.nan
    mins, secs = x.split(":")
    return int(mins) * 60 + int(secs)

df_clean["game_clock_sec"] = df_clean["game_clock"].apply(clock_to_seconds)

# drop missing shot clock values
df_clean = df_clean.dropna(subset=["shot_clock"])

# convert relevant columns to numeric types
num_cols = [
    "shot_clock",
    "dribbles",
    "touch_time",
    "shot_dist",
    "close_def_dist",
    "final_margin",
]

for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# filter out impossible or invalid values
df_clean = df_clean[(df_clean["shot_clock"] >= 0) & (df_clean["shot_clock"] <= 24)]
df_clean = df_clean[df_clean["shot_dist"] >= 0]
df_clean = df_clean[df_clean["close_def_dist"] >= 0]
df_clean = df_clean[df_clean["touch_time"] >= 0]
df_clean = df_clean[df_clean["dribbles"] >= 0]

# remove any rows missing the target variable
df_clean = df_clean.dropna(subset=["made"])

# keep only relevant features for modeling
df_clean = df_clean[
    [
        "player_name",
        "matchup",
        "location",
        "period",
        "game_clock_sec",
        "shot_clock",
        "dribbles",
        "touch_time",
        "shot_dist",
        "close_def_dist",
        "pts_type",
        "made",
    ]
]

# check remaining missing values
df_clean.isna().sum()

# save cleaned dataset
df_clean.to_csv("data/processed/shot_logs_cleaned.csv", index=False)
print("Saved to data/processed/shot_logs_cleaned.csv")

Saved to data/processed/shot_logs_cleaned.csv


In [7]:
suspect_threes = df_clean[
    (df_clean["pts_type"] == 3) &
    (df_clean["shot_dist"] < 22)
]

print(len(suspect_threes))

suspect_threes["shot_dist"].value_counts().sort_index()

weird_twos = df_clean[
    (df_clean["pts_type"] == 2) &
    (df_clean["shot_dist"] >= 24)
]

print(len(weird_twos))
weird_twos["shot_dist"].value_counts().sort_index()

774
385


shot_dist
24.0    73
24.1    54
24.2    39
24.3    22
24.4    27
24.5    14
24.6    15
24.7    10
24.8     8
24.9    14
25.0     9
25.1     5
25.2     2
25.3     6
25.4     8
25.5     3
25.6     6
25.7     4
25.8     5
25.9     3
26.1     3
26.2     3
26.3     1
26.4     2
26.5     2
26.8     4
26.9     1
27.0     2
27.1     2
27.2     1
27.3     1
27.4     2
27.5     1
27.6     2
27.7     1
27.8     1
28.0     3
28.1     2
28.2     1
28.3     2
28.4     1
28.5     3
28.6     2
28.7     2
28.9     2
29.1     1
29.3     1
29.6     2
29.7     1
34.7     1
35.3     1
36.3     1
36.5     1
43.3     1
43.5     1
Name: count, dtype: int64